<a href="https://colab.research.google.com/github/EvansAmarh/CausalMedia-GH/blob/main/cleanedEdnetKT1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import os
import random
from google.colab import drive

drive.mount('/content/drive')
print(" Drive mounted!")

Mounted at /content/drive
 Drive mounted!


In [ ]:
ednet_folder = "/content/drive/MyDrive/ednet"

files = [f for f in os.listdir(ednet_folder) if f.endswith(".csv")]
print(f"Total files found: {len(files):,}")

random.seed(42)
files_to_load = random.sample(files, min(23476, len(files)))
print(f"Will load: {len(files_to_load):,} files")

all_dfs = []
failed = 0

for i, filename in enumerate(files_to_load):
    student_id = filename.replace(".csv", "")
    filepath = os.path.join(ednet_folder, filename)
    try:
        df = pd.read_csv(filepath, header=None,
                         names=["timestamp", "solving_id", "question_id",
                                "user_answer", "elapsed_time", "is_correct"])
        df["student_id"] = student_id
        all_dfs.append(df)
    except:
        failed += 1

    if (i + 1) % 5000 == 0:
        print(f"  Loaded {i+1:,}/{len(files_to_load):,} files...")

raw_df = pd.concat(all_dfs, ignore_index=True)
print(f"\n Loaded {len(all_dfs):,} students | Failed: {failed}")
print(f"Total rows: {raw_df.shape[0]:,}")

# Save immediately so you never reload again!
raw_df.to_csv("/content/drive/MyDrive/raw_ednet_50k.csv", index=False)
print(" Raw file saved → raw_ednet_50k.csv")

Total files found: 23,478
Will load: 23,476 files
  Loaded 5,000/23,476 files...
  Loaded 10,000/23,476 files...
  Loaded 15,000/23,476 files...
  Loaded 20,000/23,476 files...

 Loaded 23,476 students | Failed: 0
Total rows: 9,731,401
 Raw file saved → raw_ednet_50k.csv


In [ ]:
# Fix timestamps — ms to real dates
raw_df["timestamp"] = pd.to_datetime(raw_df["timestamp"], unit="ms", errors="coerce")
raw_df = raw_df.sort_values(["student_id", "timestamp"])

# Build day number per student
raw_df["day"] = raw_df.groupby("student_id")["timestamp"].transform(
    lambda x: (x - x.min()).dt.days + 1
)

print("Date range:", raw_df["timestamp"].min(), "→", raw_df["timestamp"].max())
print("Day range:", raw_df["day"].min(), "to", raw_df["day"].max())
print("\nDay value counts (first 10):")
print(raw_df["day"].value_counts().sort_index().head(10))

/tmp/ipykernel_43332/3203657609.py:2: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  raw_df["timestamp"] = pd.to_datetime(raw_df["timestamp"], unit="ms", errors="coerce")


Date range: 2017-04-18 11:04:56.900000 → 2019-12-02 17:01:34.437000
Day range: 1.0 to 926.0

Day value counts (first 10):
day
1.0     503425
2.0     125187
3.0     134724
4.0     106366
5.0      99752
6.0      98854
7.0      95189
8.0      93685
9.0      88721
10.0     83501
Name: count, dtype: int64


In [ ]:
temp_df = raw_df.dropna(subset=["question_id", "user_answer"])

correct_answers = (temp_df.groupby("question_id")["user_answer"]
                   .agg(lambda x: x.mode()[0] if not x.empty else pd.NA)
                   .rename("correct_answer"))


raw_df = raw_df.merge(correct_answers, on="question_id", how="left")

# Now calculate is_correct, which might still be pd.NA if correct_answer was pd.NA
raw_df["is_correct"] = (raw_df["user_answer"] == raw_df["correct_answer"]).astype("Int64")
raw_df = raw_df.drop(columns=["correct_answer"])

print("is_correct value counts:")
print(raw_df["is_correct"].value_counts(dropna=False)) # Include NaN counts to see if any are left
print(f"Accuracy rate: {raw_df['is_correct'].mean():.1%}")

# Clean elapsed_time
raw_df["elapsed_time"] = pd.to_numeric(raw_df["elapsed_time"], errors="coerce")
raw_df = raw_df[(raw_df["elapsed_time"] > 0) & (raw_df["elapsed_time"] < 300000)]

# Drop any remaining NaNs
# This will also drop rows where is_correct became pd.NA because the correct_answer could not be determined
raw_df = raw_df.dropna(subset=["student_id", "timestamp", "elapsed_time", "is_correct"])

print(f"\n Clean rows: {raw_df.shape[0]:,}")

is_correct value counts:
is_correct
1    6799783
0    2931618
Name: count, dtype: Int64
Accuracy rate: 69.9%

 Clean rows: 9,657,390


In [ ]:
# Count interactions per student
counts = raw_df.groupby("student_id").size()
print(f"Total students before filter: {len(counts):,}")
print(f"Interaction count stats:\n{counts.describe()}")

# Keep only students with 15+ interactions
eligible = counts[counts >= 15].index
print(f"\nStudents with ≥15 interactions: {len(eligible):,}")

# Sample 10,000
np.random.seed(42)
sampled_ids = np.random.choice(eligible, size=min(10000, len(eligible)), replace=False)
df = raw_df[raw_df["student_id"].isin(sampled_ids)].copy()

print(f"\n✅ Sampled {df['student_id'].nunique():,} students")
print(f"   Rows to process: {len(df):,}")
print(f"   Sample student IDs: {list(sampled_ids[:5])}")

Total students before filter: 23,473
Interaction count stats:
count    23473.000000
mean       411.425468
std       1424.354778
min          1.000000
25%          7.000000
50%         20.000000
75%         30.000000
max      39695.000000
dtype: float64

Students with ≥15 interactions: 13,136

✅ Sampled 10,000 students
   Rows to process: 7,277,401
   Sample student IDs: ['u36837', 'u181779', 'u542980', 'u156645', 'u471458']


In [ ]:
GLOBAL_MEDIAN = df["elapsed_time"].median()
print(f"Global median elapsed time: {GLOBAL_MEDIAN:.0f} ms")

# Treatment (T) — multimedia engagement
T = (df.groupby("student_id")["elapsed_time"]
     .apply(lambda x: (x > GLOBAL_MEDIAN).mean())
     .rename("multimedia_engagement"))

# Outcome (Y) — performance gain
early_acc = df[df["day"].between(1,10)].groupby("student_id")["is_correct"].mean()
late_acc  = df[df["day"].between(21,30)].groupby("student_id")["is_correct"].mean()
Y = (late_acc - early_acc).rename("performance_gain")

# Confounders
prior_achievement = (df[df["day"].between(1,7)]
                     .groupby("student_id")["is_correct"].mean()
                     .rename("prior_achievement"))

daily_acc = (df.groupby(["student_id","day"])["is_correct"]
             .mean().reset_index()
             .rename(columns={"is_correct":"daily_acc"}))
consistency = (daily_acc.groupby("student_id")["daily_acc"]
               .std().fillna(0))
consistency = (consistency / consistency.max()).rename("consistency")

early_struggle = (df[df["day"].between(1,3)]
                  .groupby("student_id")["is_correct"].mean()
                  .rename("early_struggle"))

skill_coverage       = df.groupby("student_id")["question_id"].nunique().rename("skill_coverage")
session_duration_avg = df.groupby("student_id")["elapsed_time"].mean().rename("session_duration_avg")
total_interactions   = df.groupby("student_id").size().rename("total_interactions")
active_days = df[df['day'] <= 30].groupby('student_id')['day'].nunique()
peer_collaboration = (active_days / 30).clip(upper=1.0).rename('peer_collaboration')


student_df = (pd.DataFrame(index=sampled_ids)
              .join(T).join(Y).join(prior_achievement)
              .join(consistency).join(early_struggle)
              .join(skill_coverage).join(session_duration_avg)
              .join(total_interactions).join(peer_collaboration))

student_df.index.name = "student_id"
student_df = student_df.dropna().reset_index()

print(f" Final shape: {student_df.shape}")
print(student_df.head(3).to_string())

Global median elapsed time: 22000 ms
 Final shape: (1838, 10)
  student_id  multimedia_engagement  performance_gain  prior_achievement  consistency  early_struggle  skill_coverage  session_duration_avg  total_interactions  peer_collaboration
0    u181779               0.402385          0.281189                0.4     0.157406             0.4            5364          21785.592723                6541            0.533333
1     u21788               0.527881         -0.276923           0.736559       0.2292        0.710526             267          33594.795539                 269            0.366667
2       u115               0.626595         -0.092282           0.438017     0.152085        0.403226            1347          33812.496306                1489            0.400000


In [ ]:
print("=== SHAPE ===")
print(student_df.shape)

print("\n=== MISSING VALUES (all should be 0) ===")
print(student_df.isnull().sum())

print("\n=== KEY CHECKS ===")
checks = [
    ("multimedia_engagement", 0,  1,   0.01),
    ("performance_gain",     -1,  1,   0.01),
    ("prior_achievement",     0,  1,   0.01),
    ("early_struggle",        0,  1,   0.05),
    ("consistency",           0,  0.5, 0.01),
]
all_pass = True
for col, lo, hi, min_std in checks:
    mn  = student_df[col].min()
    mx  = student_df[col].max()
    avg = student_df[col].mean()
    std = student_df[col].std()
    ok  = "" if mn >= lo and mx <= hi and std >= min_std else " PROBLEM"
    if "" in ok: all_pass = False
    print(f"  {col}: mean={avg:.3f}, std={std:.3f}, min={mn:.3f}, max={mx:.3f}  {ok}")

print("\n" + (" ALL PASSED — safe to save!" if all_pass
              else " DO NOT SAVE — fix problems first!"))

=== SHAPE ===
(1838, 10)

=== MISSING VALUES (all should be 0) ===
student_id               0
multimedia_engagement    0
performance_gain         0
prior_achievement        0
consistency              0
early_struggle           0
skill_coverage           0
session_duration_avg     0
total_interactions       0
peer_collaboration       0
dtype: int64

=== KEY CHECKS ===
  multimedia_engagement: mean=0.480, std=0.186, min=0.000, max=0.966  
  performance_gain: mean=0.032, std=0.160, min=-0.789, max=0.867  
  prior_achievement: mean=0.624, std=0.141, min=0.000, max=1.000  
  early_struggle: mean=0.610, std=0.148, min=0.000, max=1.000  
  consistency: mean=0.234, std=0.096, min=0.000, max=0.846   PROBLEM

 DO NOT SAVE — fix problems first!


In [ ]:
output_path = "/content/drive/MyDrive/EdnetKT_clean.csv"
student_df.to_csv(output_path, index=False)

print(f" Saved → {output_path}")
print(f"   Rows:    {len(student_df)}")
print(f"   Columns: {list(student_df.columns)}")

 Saved → /content/drive/MyDrive/EdnetKT_clean.csv
   Rows:    1838
   Columns: ['student_id', 'multimedia_engagement', 'performance_gain', 'prior_achievement', 'consistency', 'early_struggle', 'skill_coverage', 'session_duration_avg', 'total_interactions', 'peer_collaboration']
